# 28 Reference-Free Preference Optimization 如何避免参考模型？

## 面试回答主线

reference-free preference optimization 的目标是避免像 DPO 一样为每个 chosen/rejected 还要跑一份 reference policy。以 SimPO 为例，可直接把长度归一的 policy log-prob 当作隐式 reward，并使用 margin 的 Bradley–Terry 偏好目标。省 reference 降低显存与算力，但也失去一个显式锚点，需更谨慎监控长度、过拟合与质量退化。实验对六条客服 prompt 的 chosen/rejected logprob 比较 reference-based margin 和长度归一的 reference-free margin，并构造使用 logprob sum 导致短回答偏置的失败。

**核心公式：** SimPO 形式可写为 $r_\theta(x,y)=\beta\log\pi_\theta(y|x)/|y|$，损失为 $-\log\sigma(r_w-r_l-\gamma)$；它不需要额外 reference model 前向。

后续依次展示同数据基线、手写核心状态/概率、结果表、真实失败与修复。数值仅用于机制验证。


## 真实案例

数据是六条脱敏客服 prompt，每条含 chosen/rejected 回答；注意力主题会将它们映射成流式键值事件。字段语义和失败模式与真实系统一致，但样本规模不能代表线上效果。


In [1]:
import math  # 导入数学函数实现概率和复杂度公式。
import warnings  # 导入警告控制模块保持输出干净。
warnings.filterwarnings('ignore', message='The pynvml package is deprecated')  # 屏蔽环境依赖产生的弃用提示。
import torch  # 导入张量和自动微分基础能力。
import torch.nn as nn  # 导入模块基类以显式定义网络。
torch.manual_seed(41)  # 固定随机种子保证输出可复现。
torch.set_num_threads(1)  # 固定小实验 CPU 线程数。
samples = [  # 定义六条可读的 prompt、候选回复或流式事件。
    {'id': 'P01', 'prompt': '支付重复扣款怎么处理？', 'chosen': '核验订单后原路退款。', 'rejected': '无需核验直接忽略。'},  # 退款决策样本。
    {'id': 'P02', 'prompt': '发现陌生转账怎么办？', 'chosen': '立即冻结并核验身份。', 'rejected': '等待下个账单周期。'},  # 账户安全样本。
    {'id': 'P03', 'prompt': '收不到登录验证码？', 'chosen': '检查手机号并重发。', 'rejected': '建议注销账户。'},  # 登录支持样本。
    {'id': 'P04', 'prompt': '地址如何修改？', 'chosen': '在发货前更新地址。', 'rejected': '永久不可修改。'},  # 售后样本。
    {'id': 'P05', 'prompt': '银行卡被盗刷？', 'chosen': '冻结卡并保留证据。', 'rejected': '继续正常使用。'},  # 风险样本。
    {'id': 'P06', 'prompt': '发票抬头写错？', 'chosen': '按规则更正抬头。', 'rejected': '删除全部订单。'},  # 账单样本。
]  # 结束可读数据定义。
print('教学实验：六条脱敏客服 prompt/候选或流式状态，只解释训练与状态机制。')  # 声明实验边界。
for row in samples:  # 逐条展示 prompt/chosen/rejected。
    print(f"{row['id']} | 问题={row['prompt']} | chosen={row['chosen']} | rejected={row['rejected']}")  # 输出真实语义样本。


教学实验：六条脱敏客服 prompt/候选或流式状态，只解释训练与状态机制。
P01 | 问题=支付重复扣款怎么处理？ | chosen=核验订单后原路退款。 | rejected=无需核验直接忽略。
P02 | 问题=发现陌生转账怎么办？ | chosen=立即冻结并核验身份。 | rejected=等待下个账单周期。
P03 | 问题=收不到登录验证码？ | chosen=检查手机号并重发。 | rejected=建议注销账户。
P04 | 问题=地址如何修改？ | chosen=在发货前更新地址。 | rejected=永久不可修改。
P05 | 问题=银行卡被盗刷？ | chosen=冻结卡并保留证据。 | rejected=继续正常使用。
P06 | 问题=发票抬头写错？ | chosen=按规则更正抬头。 | rejected=删除全部订单。


## Baseline / 基线

先运行最朴素、但同样使用这些输入和同一指标的对照，避免只看一个核心算法数字。


In [2]:
pairs = [{'prompt': '退款', 'chosen_logp': -4.0, 'rejected_logp': -5.0, 'ref_chosen': -4.5, 'ref_rejected': -5.1, 'chosen_len': 5, 'rejected_len': 4}, {'prompt': '盗刷', 'chosen_logp': -5.0, 'rejected_logp': -4.7, 'ref_chosen': -5.6, 'ref_rejected': -4.9, 'chosen_len': 6, 'rejected_len': 3}, {'prompt': '验证码', 'chosen_logp': -3.0, 'rejected_logp': -3.8, 'ref_chosen': -3.4, 'ref_rejected': -4.2, 'chosen_len': 3, 'rejected_len': 4}, {'prompt': '地址', 'chosen_logp': -4.2, 'rejected_logp': -4.9, 'ref_chosen': -4.8, 'ref_rejected': -5.1, 'chosen_len': 5, 'rejected_len': 4}, {'prompt': '冻结', 'chosen_logp': -5.4, 'rejected_logp': -6.1, 'ref_chosen': -5.8, 'ref_rejected': -6.4, 'chosen_len': 6, 'rejected_len': 5}, {'prompt': '发票', 'chosen_logp': -3.5, 'rejected_logp': -4.0, 'ref_chosen': -3.9, 'ref_rejected': -4.3, 'chosen_len': 4, 'rejected_len': 3}]  # 构造六条有 chosen/rejected 的偏好对统计。
dpo_margins = [(row['chosen_logp'] - row['rejected_logp']) - (row['ref_chosen'] - row['ref_rejected']) for row in pairs]  # 计算需要 reference 的 DPO 相对 margin。
baseline_metric = sum(dpo_margins) / len(dpo_margins)  # 保存平均 reference-based margin。
print(f'DPO reference margins={ [round(value, 3) for value in dpo_margins] }，均值={baseline_metric:.3f}')  # 展示基线需要两套模型 logprob。


DPO reference margins=[0.4, 0.4, -0.0, 0.4, 0.1, 0.1]，均值=0.233


## 手写核心实现与中间量

核心实现保留 state、ratio、优势、mask 或概率分母等中间量，不用 Trainer 或现成 Agent/Attention 框架遮蔽机制。


In [3]:
beta = 2.0  # 设置 reference-free 隐式 reward 的温度系数。
target_margin = 0.15  # 设置偏好目标间隔。
simpo_margins = [beta * (row['chosen_logp'] / row['chosen_len'] - row['rejected_logp'] / row['rejected_len']) - target_margin for row in pairs]  # 手写长度归一的 SimPO margin。
simpo_losses = [-math.log(1.0 / (1.0 + math.exp(-margin))) for margin in simpo_margins]  # 手写 Bradley-Terry logistic loss。
core_metric = sum(simpo_losses) / len(simpo_losses)  # 记录 reference-free 平均偏好损失。
print(f'SimPO margins={ [round(value, 3) for value in simpo_margins] }，losses={ [round(value, 3) for value in simpo_losses] }，均值={core_metric:.3f}')  # 展示不需要 reference 的核心中间量。


SimPO margins=[0.75, 1.317, -0.25, 0.62, 0.49, 0.767]，losses=[0.387, 0.237, 0.826, 0.43, 0.478, 0.382]，均值=0.457


In [4]:
comparison_rows = [('Baseline', float(baseline_metric)), ('核心机制', float(core_metric))]  # 建立基线与核心的同口径结果表。
for name, metric in comparison_rows:  # 逐行输出结果表。
    print(f'{name:<8} | 指标={metric:.6f}')  # 显示可读数值对照。


Baseline | 指标=0.233333
核心机制     | 指标=0.456690


## 结果解读

这里只能得出本受控样本上的机制结论。生产要用独立质量集、长度控制指标与安全评测补足 reference 缺失的约束；reference-free 并不等于不需要 KL/漂移监控。 生产决策必须进一步看验证集、线上安全指标、算力和版本可追溯性。

## 失败案例

下方先让关键条件真实失效，再展示修复如何改变可观测指标。


In [5]:
short_pair = pairs[1]  # 选择 chosen 更长、rejected 更短的盗刷回答对。
failure_metric = beta * (short_pair['chosen_logp'] - short_pair['rejected_logp'])  # 错误地用总 logprob 计算隐式 reward margin。
fix_metric = beta * (short_pair['chosen_logp'] / short_pair['chosen_len'] - short_pair['rejected_logp'] / short_pair['rejected_len'])  # 修复为按长度平均 logprob。
print(f'失败：总 logprob margin={failure_metric:.3f}；修复：长度归一 margin={fix_metric:.3f}')  # 展示 response length 会改变 reference-free 偏好。


失败：总 logprob margin=-0.600；修复：长度归一 margin=1.467


## 工程取舍、常见坑与延伸追问

**工程取舍：** 生产要用独立质量集、长度控制指标与安全评测补足 reference 缺失的约束；reference-free 并不等于不需要 KL/漂移监控。

**常见坑：** 用总 logprob 而非长度平均，把模型偏好误解为 reward，或因为省了一次前向就跳过可靠评测。

**延伸追问：** SimPO 的 target margin 如何影响输出长度？reference-free 如何与 KL controller 或 early-stop 配合？

## 生产差距

实验运行于 CPU/FP32，只有 6 条离线样本，省略了真实 rollout、分布式同步、混合精度、内容安全、数据治理、checkpoint 和监控。上线版本应以受审计的状态、指标和回滚流程替代这些教学变量。


In [6]:
assert len(simpo_losses) == 6  # 验证六条偏好对都计算了 reference-free loss。
assert core_metric > 0.0  # 验证 logistic preference loss 有效。
assert abs(failure_metric - fix_metric) > 1e-3  # 验证不归一化会改变长度不等 pair 的 margin。
assert baseline_metric != 0.0  # 验证 DPO 基线包含 reference 相对变化。
